# Strategy scan: all rules x all assets

Run every strategy in `sysstrat` on all five assets, then look at the picture from three angles: per-asset Sharpe, the diversified equal-weight portfolio per strategy, and signal correlations.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sysstrat import (
    Asset, Capital, FixedRiskSizer, BacktestRunner, PortfolioRunner,
    BuyAndHoldStrategy, MACrossoverStrategy, EWMACStrategy, NormalisedTrendStrategy,
    BreakoutStrategy, AccelerationStrategy, SkewStrategy, MeanReversionStrategy,
    TimeSeriesMomentumStrategy, DonchianStrategy, BollingerMeanReversionStrategy,
    MACDStrategy, RSIMeanReversionStrategy, FixedSignalStrategy,
    cross_sectional_momentum, cross_sectional_reversal,
    load_simple_price_csv, print_comparison_table,
)

In [ ]:
# Locate the research repo root (contains data/MCFTR.csv)
ROOT = Path.cwd()
while not (ROOT / "data" / "MCFTR.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
print(f"Data dir: {DATA_DIR}")

In [ ]:
INSTRUMENTS = {
    "MCFTR":  "MCFTR.csv",       # broad equity index
    "RGBITR": "RGBITR.csv",      # government bond index (total return)
    "GLDRUB": "GLDRUB_TOM.csv",  # gold, in RUB
    "CNYRUB": "CNYRUB_TOM.csv",  # CNY/RUB FX
    "USDRUB": "USDRUB.csv",      # USD/RUB FX
}

assets = {
    t: Asset(ticker=t, price_data=load_simple_price_csv(DATA_DIR / f),
             commission_rate=0.0004, slippage_rate=0.001)
    for t, f in INSTRUMENTS.items()
}

start = max(a.price_data.index.min() for a in assets.values())
end = min(a.price_data.index.max() for a in assets.values())
assets = {t: a.slice(start, end) for t, a in assets.items()}
print(f"Common period: {start.date()} -> {end.date()}")

In [ ]:
capital = Capital(initial_capital=100_000)
sizer = FixedRiskSizer(risk_target=0.20, max_leverage=1.0)

In [ ]:
STRATEGIES = {
    "Buy & Hold":          BuyAndHoldStrategy(),
    "MA Cross (10/50)":    MACrossoverStrategy(short_window=10, long_window=50),
    "MA Cross LS":         MACrossoverStrategy(short_window=10, long_window=50, mode="long_short"),
    "EWMAC (16/64)":       EWMACStrategy(),
    "EWMAC (32/128)":      EWMACStrategy(fast_window=32, slow_window=128),
    "Normalised Trend":    NormalisedTrendStrategy(),
    "Breakout (40)":       BreakoutStrategy(horizon=40),
    "Breakout (160)":      BreakoutStrategy(horizon=160),
    "Acceleration":        AccelerationStrategy(),
    "Skew":                SkewStrategy(),
    "Mean Reversion":      MeanReversionStrategy(),
    "TSMOM (252/21)":      TimeSeriesMomentumStrategy(),
    "Donchian (55/20)":    DonchianStrategy(),
    "Bollinger MR (40,2)": BollingerMeanReversionStrategy(),
    "MACD (12/26/9)":      MACDStrategy(),
    "RSI(2) MR":           RSIMeanReversionStrategy(),
}

## 1. Per-asset Sharpe (strategy x asset)

Each cell is a single-asset backtest, vol-targeted at 20%.

In [ ]:
sharpe = {}
total_ret = {}
for sname, strategy in STRATEGIES.items():
    sharpe[sname] = {}
    total_ret[sname] = {}
    for t, a in assets.items():
        m = BacktestRunner(capital, a, sizer).run(strategy).metrics
        sharpe[sname][t] = round(m.sharpe_ratio, 2)
        total_ret[sname][t] = round(m.total_return_pct, 1)

sharpe_df = pd.DataFrame(sharpe).T[list(assets)]
total_ret_df = pd.DataFrame(total_ret).T[list(assets)]
sharpe_df

In [ ]:
total_ret_df

## 2. Diversified equal-weight portfolio per strategy

Each strategy run across the basket and combined 1/N.

In [ ]:
rows = []
for sname, strategy in STRATEGIES.items():
    reports = {t: BacktestRunner(capital, a, sizer).run(strategy) for t, a in assets.items()}
    p = PortfolioRunner(capital).run(reports)
    m = p.metrics
    rows.append({
        "strategy": sname,
        "total ret %": round(m.total_return_pct, 1),
        "vol %": round(m.annual_volatility_pct, 2),
        "sharpe": round(m.sharpe_ratio, 2),
        "sortino": round(m.sortino_ratio, 2),
        "max DD %": round(m.max_drawdown_pct, 1),
    })

port_summary = pd.DataFrame(rows).set_index("strategy")
port_summary

## 3. Signal correlation (MCFTR)

Rules with correlation well below 1 capture different behaviour and are candidates for combination.

In [ ]:
d = assets["MCFTR"].price_data.to_frame(name="close")
signals = {name: strategy.generate_signals(d) for name, strategy in STRATEGIES.items()}

corr = pd.DataFrame({
    a: {b: round(signals[a].corr(signals[b]), 2) for b in signals}
    for a in signals
})
corr

## 4. Cross-sectional strategies

Basket-level rules: compute signals on the cross-section, wrap each column in `FixedSignalStrategy`, and run through the normal pipeline.

In [ ]:
prices = pd.DataFrame({t: a.price_data for t, a in assets.items()}).dropna()
cs_mom = cross_sectional_momentum(prices)
cs_rev = cross_sectional_reversal(prices)

cs_results = {}
for name, signals_df in [("CS Momentum", cs_mom), ("CS Reversal", cs_rev)]:
    reports = {
        t: BacktestRunner(capital, assets[t], sizer).run(
            FixedSignalStrategy(signals_df[t], name=f"{name} {t}")
        )
        for t in assets
    }
    cs_results[name] = PortfolioRunner(capital).run(reports)

bh_reports = {t: BacktestRunner(capital, a, sizer).run(BuyAndHoldStrategy()) for t, a in assets.items()}
bh_port = PortfolioRunner(capital).run(bh_reports)

print_comparison_table(
    "Cross-sectional vs equal-weight Buy & Hold",
    {
        "CS Momentum": cs_results["CS Momentum"].metrics,
        "CS Reversal": cs_results["CS Reversal"].metrics,
        "B&H equal-wt": bh_port.metrics,
    },
)